# Minimum-Variance portofolio

Assume that an investor can create a portfolio by allocating to three assets _A_, _B_ and _C_ with the following correlation matrix $\boldsymbol{C}$, vector of volatilities $\boldsymbol{v}$, and expected return vector $\boldsymbol{\mu}$

$$
\boldsymbol{C} = \begin{pmatrix} 1.0 & 0.2 & 0.15 \\
0.2 & 1.0 & 0.4 \\
0.15 & 0.4 & 1.0\end{pmatrix}
$$

and 

$$
\boldsymbol{v} = \begin{pmatrix} 0.1 \\ 0.15 \\ 0.2 \end{pmatrix}
$$


and 

$$
\boldsymbol{\mu} = \begin{pmatrix} 0.04 \\ 0.06 \\ 0.08 \end{pmatrix}
$$

In [1]:
import numpy as np

## Problem 1

Define the correlation matrix, covariance matrix and the vector of expected returns as `numpy` arrays

### Solution

In [2]:
# correlation matrix, volatilities and expected returns
corr_mat = np.array([[1.0, 0.2, 0.15],
                     [0.2, 1.0, 0.4],
                     [0.15, 0.4, 1.0]])
vols = np.array([0.10, 0.15, 0.20])
mu = np.array([0.04, 0.06, 0.08])

# covariance matrix: Sigma = diag(v) C diag(v)  (same as np.outer(vols, vols) * corr_mat)
cov_mat = np.diag(vols) @ corr_mat @ np.diag(vols)
cov_mat

array([[0.01  , 0.003 , 0.003 ],
       [0.003 , 0.0225, 0.012 ],
       [0.003 , 0.012 , 0.04  ]])

**Interpretation:** The diagonal of $\boldsymbol{\Sigma}$ holds the variances ($0.1^2 = 0.01$, $0.15^2 = 0.0225$, $0.2^2 = 0.04$); the off-diagonal elements are the covariances $\sigma_{ij} = \rho_{ij}\sigma_i\sigma_j$, e.g. $\sigma_{BC} = 0.4 \cdot 0.15 \cdot 0.20 = 0.012$.

## Problem 2

Check if the covariance matrix is postive definite. 

### Solution

In [3]:
# a symmetric matrix is positive definite <=> all eigenvalues > 0
eig_vals, _ = np.linalg.eigh(cov_mat)   # eigh: for symmetric matrices, real eigenvalues
print(eig_vals)
print('positive definite:', np.all(eig_vals > 0))

# alternative check: the Cholesky decomposition only exists for positive definite matrices
L = np.linalg.cholesky(cov_mat)

[0.00931378 0.01663894 0.04654728]
positive definite: True


**Interpretation:** All three eigenvalues (0.0093, 0.0166, 0.0465) are positive, so $\boldsymbol{\Sigma}$ is positive definite: every portfolio $\mathbf{w} \neq \mathbf{0}$ has $\mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w} > 0$, and $\boldsymbol{\Sigma}^{-1}$ exists – it is a valid covariance matrix.

## Problem 3

Consider an equally weighted portfolio $w_A = w_B = w_C = 1/3$. What is the formula for the portfolio variance using matrix notation? Calculate the portfolio standard devation for the equally weighted portfolio. Also calculate the expected return for the portfolio. 


### Solution

In [4]:
w_eq = np.repeat(1 / 3, 3)

mu_eq = w_eq @ mu                        # w'mu
std_eq = np.sqrt(w_eq @ cov_mat @ w_eq)   # sqrt(w' Sigma w)
print(f'expected return: {mu_eq:.4f}, standard deviation: {std_eq:.4f}')

expected return: 0.0600, standard deviation: 0.1098


**Interpretation:** The portfolio variance is the quadratic form $\sigma_p^2 = \mathbf{w}^\top\boldsymbol{\Sigma}\mathbf{w}$ and the expected return $\mu_p = \mathbf{w}^\top\boldsymbol{\mu}$. Equal weights give 6.00% expected return and 10.98% volatility – below the average single-asset volatility of 15% because the assets are imperfectly correlated.

## Problem 4

The portfolio weights $\mathbf{w} = (w_A \; \; w_B \; \; w_C)^\top$ satisfies the budget constraint $\mathbf{w}^\top \mathbf{1} =  w_A + w_B + w_C = 1$. The investor seeks to minimize the variance subject to this condition

$$
\min \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \; \; \;  \text{s.t. } \mathbf{w}^\top \mathbf{1}  = 1
$$

The Lagrangian reads

$$
\mathcal{L} = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} - \lambda (\mathbf{w}^\top \mathbf{1}  - 1)
$$

Deriving the FOC and solving for the optimal weights yields

$$
\mathbf{w}^* = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

Define a function which calculates the minimum-variance portfolio weights given a covariance matrix. Calculate the minimum-variance portfolio weights. 


### Solution

In [5]:
def min_variance_weights(cov_mat: np.ndarray) -> np.ndarray:
    """
    Minimum-variance weights with a budget constraint (short sales allowed):
    w* = Sigma^{-1} 1 / (1' Sigma^{-1} 1)
    """
    ones = np.ones(cov_mat.shape[0])
    x = np.linalg.solve(cov_mat, ones)   # Sigma^{-1} 1 without forming the inverse
    return x / (ones @ x)

w_mv = min_variance_weights(cov_mat)
print('weights:', w_mv.round(4))
print(f'expected return: {w_mv @ mu:.4f}, standard deviation: {np.sqrt(w_mv @ cov_mat @ w_mv):.4f}')

weights: [0.7038 0.2154 0.0808]
expected return: 0.0475, standard deviation: 0.0890


**Interpretation:** The minimum-variance portfolio holds 70.4% A, 21.5% B and 8.1% C. Risk falls from 10.98% (equal weights) to 8.90%, at the cost of a lower expected return (4.75% vs 6.00%). It only needs the covariance matrix – no expected returns – which makes it robust to errors in $\boldsymbol{\mu}$.